In [4]:
import os
import cv2 as cv
import numpy as np

# ============================================================
# STACKED VIDEO TRACKING + 3D TRIANGULATION + LOCAL TUFT ANGLE
#
# REVISION: Added motion-aided prediction module
#   - Lucas-Kanade optical flow for ROI center prediction
#   - Per-marker Kalman filter for state estimation
#   - Predict-then-correct loop replaces static-ROI baseline
#
# Stacked video:
#   TOP    = LEFT
#   BOTTOM = RIGHT
# ============================================================

# -------------------------
# INPUTS / OUTPUTS
# -------------------------
STACKED_VIDEO = "CFD_Merged_Seyir2.mp4"
CALIB_NPZ     = "stereo_calib_video2.npz"

OUT_DIR = "tracked_xyz_stacked_localtuft_original_tuft"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_TAG = "CFD_Merged_Seyir2_localtuft_kalman_lk"
OUT_NPZ = os.path.join(OUT_DIR, f"tracked_xyz_mm_{OUT_TAG}.npz")
OUT_CSV = os.path.join(OUT_DIR, f"tracked_xyz_mm_{OUT_TAG}.csv")
OUT_MP4 = os.path.join(OUT_DIR, f"tracked_overlay_{OUT_TAG}.mp4")
OUT_AVI = os.path.join(OUT_DIR, f"tracked_overlay_{OUT_TAG}.avi")

# -------------------------
# SETTINGS
# -------------------------
UPSCALE_TO_H = 1080

ANGLE_REF = "X"

# ---- RED TRACKING ----
MORPH_OPEN_IT  = 0
MORPH_CLOSE_IT = 1
MIN_AREA       = 1

AUTO_THR = True
RED_DIFF_THR_BASE = 35
RED_R_THR_BASE    = 60
V_REF = 110.0
SCALE_CLAMP = (0.60, 1.20)

ENABLE_TOP_BONUS = True
TOP_Y_FRAC = 0.35
TOP_BONUS_MULT = 0.85

RELAX_STEPS = [1.0, 1.6, 2.4, 3.0]

BASE_ROI_SIZE     = (60, 60)
ROI_EXPAND_2X     = 2.0
USED_MIN_DIST     = 30
GAP_HOLD          = 3
SET_NAN_AFTER_GAP = False
AREA_ALPHA        = 0.15

# ---- OPTICAL FLOW SETTINGS ----
USE_OPTICAL_FLOW = True
LK_WIN_SIZE      = (21, 21)   # Lucas-Kanade search window
LK_MAX_LEVEL     = 3          # pyramid levels
LK_MAX_ITER      = 20
LK_EPSILON       = 0.01
LK_MIN_EIGEN_THR = 1e-4       # below this -> discard flow prediction

# ---- KALMAN FILTER SETTINGS ----
USE_KALMAN       = True
# State: [x, y, vx, vy], Measurement: [x, y]
KALMAN_PROC_NOISE  = 5.0      # process noise (Q diagonal)
KALMAN_MEAS_NOISE  = 2.0      # measurement noise (R diagonal)
KALMAN_ERROR_COV   = 100.0    # initial error covariance (P diagonal)

# ---- LOCAL TUFT SETTINGS ----
TUFT_SEARCH_W   = 80
TUFT_SEARCH_H   = 100
TUFT_Y_OFFSET   = 0
TUFT_MIN_DOWN   = 0
TUFT_MAX_RADIUS = 90

TUFT_BLACKHAT_K = 9
TUFT_MIN_PIXELS = 5
TUFT_OPEN_IT    = 0
TUFT_DILATE_IT  = 1

ENABLE_TUFT_REACQUIRE = True
TUFT_REACQUIRE_RADIUS = 120

# ---- VIDEO RANGE ----
ASK_RANGE = True
START_FRAME_FIXED = 100
END_FRAME_FIXED   = 110

# ---- OUTPUT VIDEO ----
USE_INPUT_FPS = True
OUT_FPS   = 20
WRITE_AVI = True
AVI_CODECS = ["MJPG", "XVID", "DIVX", "I420"]

# ---- DISPLAY ----
SHOW_PHYS_ANGLE    = True
SHOW_2D_DEBUG_ANGLES = False

# ============================================================
# HELPERS
# ============================================================

def split_top_bottom(frame_bgr):
    h, w = frame_bgr.shape[:2]
    h2 = h // 2
    return frame_bgr[:h2, :], frame_bgr[h2:h2*2, :]

def maybe_upscale(img_bgr, target_h):
    if target_h is None:
        return img_bgr
    h, w = img_bgr.shape[:2]
    if h == target_h:
        return img_bgr
    s = target_h / float(h)
    new_w = int(round(w * s))
    return cv.resize(img_bgr, (new_w, target_h), interpolation=cv.INTER_CUBIC)

def safe_get_video_prop(cap, prop, default=0):
    v = cap.get(prop)
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return default
    return v

def open_video_writer_avi(path, fps, size_wh, codecs):
    for c in codecs:
        vw = cv.VideoWriter(path, cv.VideoWriter_fourcc(*c), float(fps), size_wh)
        if vw.isOpened():
            return vw, c
        vw.release()
    return None, None

def clamp_rect(rect, w, h):
    x, y, rw, rh = rect
    x = int(max(0, min(x, w - 1)))
    y = int(max(0, min(y, h - 1)))
    rw = int(max(1, min(rw, w - x)))
    rh = int(max(1, min(rh, h - y)))
    return (x, y, rw, rh)

def roi_from_center(center_xy, size_wh, frame_wh):
    cx, cy = float(center_xy[0]), float(center_xy[1])
    w, h = float(size_wh[0]), float(size_wh[1])
    x = int(round(cx - w/2))
    y = int(round(cy - h/2))
    W, H = frame_wh
    return clamp_rect((x, y, int(round(w)), int(round(h))), W, H)

def roi_center(roi):
    x, y, w, h = roi
    return (x + w/2.0, y + h/2.0)

def wrap_deg(a):
    if not np.isfinite(a):
        return np.nan
    return float((a + 180.0) % 360.0 - 180.0)

# ============================================================
# KALMAN FILTER FACTORY
# ============================================================

def make_kalman():
    """
    Create a 4-state (x, y, vx, vy) / 2-measurement (x, y) Kalman filter.
    Models near-constant velocity between frames.
    """
    kf = cv.KalmanFilter(4, 2)

    # Transition matrix: x' = x + vx, y' = y + vy
    kf.transitionMatrix = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
    ], dtype=np.float32)

    # Measurement matrix: observe x and y only
    kf.measurementMatrix = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
    ], dtype=np.float32)

    # Process noise covariance
    kf.processNoiseCov = np.eye(4, dtype=np.float32) * KALMAN_PROC_NOISE

    # Measurement noise covariance
    kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * KALMAN_MEAS_NOISE

    # Initial error covariance
    kf.errorCovPost = np.eye(4, dtype=np.float32) * KALMAN_ERROR_COV

    return kf

def kalman_init(kf, x, y):
    """Initialize Kalman state at given position with zero velocity."""
    kf.statePost = np.array([[x], [y], [0.0], [0.0]], dtype=np.float32)

def kalman_predict(kf):
    """Predict next state. Returns (x, y) prediction."""
    pred = kf.predict()
    return float(pred[0]), float(pred[1])

def kalman_correct(kf, x, y):
    """Correct with measurement. Returns corrected (x, y)."""
    meas = np.array([[x], [y]], dtype=np.float32)
    corr = kf.correct(meas)
    return float(corr[0]), float(corr[1])

# ============================================================
# OPTICAL FLOW PREDICTION
# ============================================================

LK_PARAMS = dict(
    winSize=LK_WIN_SIZE,
    maxLevel=LK_MAX_LEVEL,
    criteria=(cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT,
              LK_MAX_ITER, LK_EPSILON),
    minEigThreshold=LK_MIN_EIGEN_THR,
)

def optical_flow_predict(prev_gray, curr_gray, pts_xy):
    """
    Predict new positions of pts_xy using Lucas-Kanade optical flow.

    Args:
        prev_gray: previous grayscale frame
        curr_gray: current grayscale frame
        pts_xy: (N, 2) float32 array of previous positions

    Returns:
        predicted: (N, 2) float32 — predicted positions (NaN if flow failed)
        valid:     (N,) bool — True where flow succeeded
    """
    N = len(pts_xy)
    predicted = np.full((N, 2), np.nan, dtype=np.float32)
    valid = np.zeros(N, dtype=bool)

    if N == 0 or prev_gray is None:
        return predicted, valid

    pts = pts_xy.reshape(-1, 1, 2).astype(np.float32)
    next_pts, status, _ = cv.calcOpticalFlowPyrLK(
        prev_gray, curr_gray, pts, None, **LK_PARAMS
    )

    if next_pts is None or status is None:
        return predicted, valid

    status = status.reshape(-1)
    next_pts = next_pts.reshape(-1, 2)

    for i in range(N):
        if status[i] == 1:
            predicted[i] = next_pts[i]
            valid[i] = True

    return predicted, valid

# ============================================================
# RED MASK / TRACKING
# ============================================================

def red_mask_adaptive(patch_bgr, y_global_center, frame_h, relax=1.0):
    b, g, r = cv.split(patch_bgr)
    diff = cv.subtract(r, cv.max(b, g))

    hsv = cv.cvtColor(patch_bgr, cv.COLOR_BGR2HSV)
    v = hsv[..., 2].astype(np.float32)
    v_mean = float(np.mean(v))

    diff_thr = float(RED_DIFF_THR_BASE)
    r_thr    = float(RED_R_THR_BASE)

    if AUTO_THR:
        scale = v_mean / V_REF if V_REF > 1e-6 else 1.0
        scale = max(SCALE_CLAMP[0], min(SCALE_CLAMP[1], scale))
        diff_thr *= scale
        r_thr    *= scale

    if ENABLE_TOP_BONUS and (y_global_center < frame_h * TOP_Y_FRAC):
        diff_thr *= TOP_BONUS_MULT
        r_thr    *= TOP_BONUS_MULT

    if relax is None:
        relax = 1.0
    if relax > 1.0:
        diff_thr = max(1.0, diff_thr / float(relax))
        r_thr    = max(1.0, r_thr / float(relax))

    _, m1 = cv.threshold(diff, int(diff_thr), 255, cv.THRESH_BINARY)
    _, m2 = cv.threshold(r,    int(r_thr),    255, cv.THRESH_BINARY)
    mask = cv.bitwise_and(m1, m2)

    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (3, 3))
    mask = cv.dilate(mask, k, iterations=1)

    for _ in range(MORPH_OPEN_IT):
        mask = cv.morphologyEx(mask, cv.MORPH_OPEN, k)
    for _ in range(MORPH_CLOSE_IT):
        mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, k)

    return mask

def _filter_used(cands, used_pts, used_min_dist):
    if used_pts is None or len(used_pts) == 0:
        return cands
    used = np.asarray(used_pts, dtype=np.float32).reshape(-1, 2)
    thr2 = float(used_min_dist) ** 2
    out = []
    for (x, y, area) in cands:
        d2 = np.min(np.sum((used - np.array([x, y], dtype=np.float32))**2, axis=1))
        if d2 >= thr2:
            out.append((x, y, area))
    return out

def red_centroids_in_roi(frame_bgr, roi, used_pts=None, used_min_dist=30, relax=1.0):
    H, W = frame_bgr.shape[:2]
    x, y, rw, rh = clamp_rect(roi, W, H)
    patch = frame_bgr[y:y+rh, x:x+rw]

    y_center = y + rh * 0.5
    mask = red_mask_adaptive(patch, y_center, H, relax=relax)

    cnts, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    cands = []
    for c in cnts:
        area = float(cv.contourArea(c))
        if area < float(MIN_AREA):
            continue
        M = cv.moments(c)
        if M["m00"] == 0:
            continue
        cx = float(M["m10"] / M["m00"]) + x
        cy = float(M["m01"] / M["m00"]) + y
        cands.append((cx, cy, area))

    return _filter_used(cands, used_pts, used_min_dist)

def pick_nearest_with_area(cands, ref_xy, last_area=None, area_alpha=0.15):
    if not cands:
        return (np.nan, np.nan, 0.0)

    ref = np.asarray(ref_xy, dtype=np.float32).reshape(1, 2)
    pts = np.asarray([(c[0], c[1]) for c in cands], dtype=np.float32)
    d2 = np.sum((pts - ref)**2, axis=1)

    if last_area is None or (not np.isfinite(last_area)) or last_area <= 0 or area_alpha <= 0:
        k = int(np.argmin(d2))
        return cands[k]

    areas = np.asarray([max(c[2], 1e-6) for c in cands], dtype=np.float32)
    la = float(max(last_area, 1e-6))
    a_pen = (np.log(areas) - np.log(la))**2
    score = d2 + float(area_alpha) * a_pen * 1000.0
    k = int(np.argmin(score))
    return cands[k]

def track_roi_then_2x(frame_bgr, last_xy, last_area, base_roi_wh,
                      predicted_xy=None,
                      used_pts=None, used_min_dist=30):
    """
    Track marker using ROI-based color detection.
    If predicted_xy is provided (from optical flow), use it as ROI center
    instead of last_xy — reducing search failures under rapid deformation.
    Falls back to last_xy if prediction is invalid.
    """
    H, W = frame_bgr.shape[:2]
    frame_wh = (W, H)

    if not (np.isfinite(last_xy[0]) and np.isfinite(last_xy[1])):
        roi0 = roi_from_center((W/2, H/2), base_roi_wh, frame_wh)
        return (np.array([np.nan, np.nan], dtype=np.float32), roi0, False, "lost", np.nan)

    # Use optical flow prediction as ROI center if available
    if predicted_xy is not None and np.isfinite(predicted_xy[0]) and np.isfinite(predicted_xy[1]):
        roi_center_xy = predicted_xy
        used_flow = True
    else:
        roi_center_xy = last_xy
        used_flow = False

    roi1 = roi_from_center(roi_center_xy, base_roi_wh, frame_wh)

    for relax in RELAX_STEPS:
        c1 = red_centroids_in_roi(frame_bgr, roi1, used_pts=used_pts,
                                  used_min_dist=used_min_dist, relax=relax)
        cx, cy, a = pick_nearest_with_area(c1, roi_center_xy,
                                           last_area=last_area, area_alpha=AREA_ALPHA)
        if np.isfinite(cx) and np.isfinite(cy):
            suffix = "_lk" if used_flow else ""
            mode = ("roi" if relax == 1.0 else f"roi_relax{relax:.1f}") + suffix
            return (np.array([cx, cy], dtype=np.float32), roi1, True, mode, a)

    roi2_wh = (base_roi_wh[0] * ROI_EXPAND_2X, base_roi_wh[1] * ROI_EXPAND_2X)
    roi2 = roi_from_center(roi_center_xy, roi2_wh, frame_wh)

    for relax in RELAX_STEPS:
        c2 = red_centroids_in_roi(frame_bgr, roi2, used_pts=used_pts,
                                  used_min_dist=used_min_dist, relax=relax)
        cx, cy, a = pick_nearest_with_area(c2, roi_center_xy,
                                           last_area=last_area, area_alpha=AREA_ALPHA)
        if np.isfinite(cx) and np.isfinite(cy):
            roi_new = roi_from_center((cx, cy), base_roi_wh, frame_wh)
            suffix = "_lk" if used_flow else ""
            mode = ("roi2x" if relax == 1.0 else f"roi2x_relax{relax:.1f}") + suffix
            return (np.array([cx, cy], dtype=np.float32), roi_new, True, mode, a)

    return (np.array([np.nan, np.nan], dtype=np.float32), roi1, False, "lost", np.nan)

# ============================================================
# LOCAL TUFT DETECTION
# ============================================================

def angle_from_marker_to_point(marker_xy, point_xy, ref_axis="X"):
    mx, my = float(marker_xy[0]), float(marker_xy[1])
    px, py = float(point_xy[0]), float(point_xy[1])
    dx = px - mx
    dy = py - my
    if abs(dx) + abs(dy) < 1e-6:
        return np.nan
    if str(ref_axis).upper() == "X":
        ang = np.degrees(np.arctan2(dy, dx))
    else:
        ang = np.degrees(np.arctan2(dx, dy))
    return wrap_deg(float(ang))

def local_black_mask(patch_bgr, blackhat_k=9, open_it=0, dilate_it=1):
    gray = cv.cvtColor(patch_bgr, cv.COLOR_BGR2GRAY)
    ksz = int(blackhat_k)
    if ksz % 2 == 0:
        ksz += 1
    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (ksz, ksz))
    bh = cv.morphologyEx(gray, cv.MORPH_BLACKHAT, k)
    _, bw = cv.threshold(bh, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    k3 = cv.getStructuringElement(cv.MORPH_ELLIPSE, (3, 3))
    for _ in range(int(open_it)):
        bw = cv.morphologyEx(bw, cv.MORPH_OPEN, k3)
    for _ in range(int(dilate_it)):
        bw = cv.dilate(bw, k3, iterations=1)
    return bw

def find_local_tuft_centroid(frame_bgr, marker_xy,
                             search_w=80, search_h=100, y_offset=0,
                             min_down=0, max_radius=50,
                             blackhat_k=9, min_pixels=5,
                             open_it=0, dilate_it=1,
                             ref_axis="X"):
    H, W = frame_bgr.shape[:2]
    mx, my = float(marker_xy[0]), float(marker_xy[1])

    if not (np.isfinite(mx) and np.isfinite(my)):
        return (np.array([np.nan, np.nan], np.float32),
                np.nan, False, (0, 0, 1, 1), None)

    x0 = int(round(mx - search_w / 2))
    y0 = int(round(my + y_offset))
    x1 = x0 + int(search_w)
    y1 = y0 + int(search_h)

    x0 = max(0, min(x0, W - 1))
    y0 = max(0, min(y0, H - 1))
    x1 = max(x0 + 1, min(x1, W))
    y1 = max(y0 + 1, min(y1, H))

    patch = frame_bgr[y0:y1, x0:x1]
    bw = local_black_mask(patch, blackhat_k=blackhat_k,
                          open_it=open_it, dilate_it=dilate_it)

    mx_p = float(mx - x0)
    my_p = float(my - y0)

    yy_all, xx_all = np.where(bw > 0)
    if len(xx_all) < int(min_pixels):
        return (np.array([np.nan, np.nan], np.float32),
                np.nan, False, (x0, y0, x1-x0, y1-y0), bw)

    keep = yy_all >= (my_p + float(min_down))
    if not np.any(keep):
        return (np.array([np.nan, np.nan], np.float32),
                np.nan, False, (x0, y0, x1-x0, y1-y0), bw)

    bw2 = np.zeros_like(bw)
    bw2[yy_all[keep], xx_all[keep]] = 255

    num_labels, labels, stats, centroids = cv.connectedComponentsWithStats(bw2, connectivity=8)

    best_label = -1
    best_score = np.inf

    for lab in range(1, num_labels):
        area = int(stats[lab, cv.CC_STAT_AREA])
        if area < int(min_pixels):
            continue
        ys, xs = np.where(labels == lab)
        if len(xs) < int(min_pixels):
            continue
        dx = xs.astype(np.float32) - mx_p
        dy = ys.astype(np.float32) - my_p
        dist = np.sqrt(dx * dx + dy * dy)
        dmin = float(np.min(dist))
        if dmin > float(max_radius):
            continue
        score = dmin + 0.01 * area
        if score < best_score:
            best_score = score
            best_label = lab

    if best_label < 0:
        return (np.array([np.nan, np.nan], np.float32),
                np.nan, False, (x0, y0, x1-x0, y1-y0), bw2)

    ys, xs = np.where(labels == best_label)
    if len(xs) < int(min_pixels):
        return (np.array([np.nan, np.nan], np.float32),
                np.nan, False, (x0, y0, x1-x0, y1-y0), bw2)

    dx = xs.astype(np.float32) - mx_p
    dy = ys.astype(np.float32) - my_p
    dist = np.sqrt(dx * dx + dy * dy)
    k_far = int(np.argmax(dist))
    tip_global = np.array([xs[k_far] + x0, ys[k_far] + y0], dtype=np.float32)
    ang = angle_from_marker_to_point(marker_xy, tip_global, ref_axis=ref_axis)

    dbg = np.zeros_like(bw2)
    dbg[labels == best_label] = 255

    return tip_global, ang, True, (x0, y0, x1-x0, y1-y0), dbg

def find_local_tuft_with_reacquire(frame_bgr, marker_xy, ref_axis="X"):
    tip, ang, ok, search_box, dbg = find_local_tuft_centroid(
        frame_bgr, marker_xy,
        search_w=TUFT_SEARCH_W, search_h=TUFT_SEARCH_H,
        y_offset=TUFT_Y_OFFSET, min_down=TUFT_MIN_DOWN,
        max_radius=TUFT_MAX_RADIUS, blackhat_k=TUFT_BLACKHAT_K,
        min_pixels=TUFT_MIN_PIXELS, open_it=TUFT_OPEN_IT,
        dilate_it=TUFT_DILATE_IT, ref_axis=ref_axis
    )
    if ok or (not ENABLE_TUFT_REACQUIRE):
        return tip, ang, ok, search_box, dbg

    tip2, ang2, ok2, search_box2, dbg2 = find_local_tuft_centroid(
        frame_bgr, marker_xy,
        search_w=TUFT_SEARCH_W, search_h=TUFT_SEARCH_H,
        y_offset=TUFT_Y_OFFSET, min_down=TUFT_MIN_DOWN,
        max_radius=TUFT_REACQUIRE_RADIUS, blackhat_k=TUFT_BLACKHAT_K,
        min_pixels=TUFT_MIN_PIXELS, open_it=TUFT_OPEN_IT,
        dilate_it=TUFT_DILATE_IT, ref_axis=ref_axis
    )
    return tip2, ang2, ok2, search_box2, dbg2

def draw_tuft_overlay(img, marker_xy, tip_xy, ok, search_box=None, text=None):
    if search_box is not None:
        x, y, w, h = search_box
        cv.rectangle(img, (x, y), (x+w, y+h), (0, 165, 255), 2)
    if not ok:
        return
    mx, my = int(round(marker_xy[0])), int(round(marker_xy[1]))
    tx, ty = int(round(tip_xy[0])), int(round(tip_xy[1]))
    cv.line(img, (mx, my), (tx, ty), (255, 255, 0), 2)
    cv.circle(img, (tx, ty), 4, (0, 255, 255), -1)
    if text is not None:
        cv.putText(img, text, (tx+6, ty+6), cv.FONT_HERSHEY_SIMPLEX,
                   0.55, (255, 255, 0), 2)

# ============================================================
# TRIANGULATION
# ============================================================

def _rectified_newK_from_P(P):
    P = np.asarray(P)
    if P.shape == (3, 4):
        return P[:, :3].copy()
    if P.shape == (3, 3):
        return P.copy()
    raise ValueError(f"Unexpected P shape: {P.shape}")

def rectified_points_from_pixel(pts_px, K, dist, Rrect, Prect):
    pts = np.asarray(pts_px, dtype=np.float32).reshape(-1, 1, 2)
    newK = _rectified_newK_from_P(Prect)
    und = cv.undistortPoints(pts, K, dist, R=Rrect, P=newK)
    return und.reshape(-1, 2)

def triangulate_rectified(P1, P2, pts1_rect_px, pts2_rect_px):
    p1 = np.asarray(pts1_rect_px, dtype=np.float64).T
    p2 = np.asarray(pts2_rect_px, dtype=np.float64).T
    X4 = cv.triangulatePoints(P1, P2, p1, p2)
    X = (X4[:3] / X4[3]).T
    return X

def physical_angle_xy(marker_xyz, tip_xyz, ref_axis="Y"):
    if not (np.all(np.isfinite(marker_xyz)) and np.all(np.isfinite(tip_xyz))):
        return np.nan
    vx = float(tip_xyz[0] - marker_xyz[0])
    vy = float(tip_xyz[1] - marker_xyz[1])
    if abs(vx) + abs(vy) < 1e-12:
        return np.nan
    if str(ref_axis).upper() == "X":
        ang = np.degrees(np.arctan2(vy, vx))
    else:
        ang = np.degrees(np.arctan2(vx, vy))
    return wrap_deg(ang)

def draw_overlay_one(draw, roi, pt, idx, ok, mode, xyz=None):
    x, y, w, h = clamp_rect(roi, draw.shape[1], draw.shape[0])
    cv.rectangle(draw, (x, y), (x+w, y+h), (255, 255, 0), 2)
    if ok and np.isfinite(pt[0]) and np.isfinite(pt[1]):
        u, v = int(round(pt[0])), int(round(pt[1]))
        cv.circle(draw, (u, v), 5, (0, 255, 0), -1)
        cv.putText(draw, f"#{idx+1} {mode}", (u+6, v-6),
                   cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        if xyz is not None and np.all(np.isfinite(xyz)):
            X, Y, Z = xyz
            cv.putText(draw, f"X{X:.1f} Y{Y:.1f} Z{Z:.1f} mm",
                       (u+6, v+18), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    else:
        cv.putText(draw, f"#{idx+1} {str(mode).upper()}",
                   (x, max(15, y-5)), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

# ============================================================
# MAIN
# ============================================================

def main():
    if not os.path.exists(CALIB_NPZ):
        raise FileNotFoundError(f"Calibration dosyası yok: {CALIB_NPZ}")

    cal = np.load(CALIB_NPZ, allow_pickle=True)
    K1 = cal["K1"]; dist1 = cal["dist1"].reshape(-1, 1)
    K2 = cal["K2"]; dist2 = cal["dist2"].reshape(-1, 1)
    R1 = cal["R1"]; R2 = cal["R2"]
    P1 = cal["P1"]; P2 = cal["P2"]
    frameSize_calib = tuple(cal["frameSize"].astype(int))
    print("[calib] loaded | frameSize(each view w,h):", frameSize_calib)

    cap = cv.VideoCapture(STACKED_VIDEO)
    if not cap.isOpened():
        raise RuntimeError(f"Video açılamadı: {STACKED_VIDEO}")

    total_frames = int(safe_get_video_prop(cap, cv.CAP_PROP_FRAME_COUNT, 0))
    fps_in = float(safe_get_video_prop(cap, cv.CAP_PROP_FPS, 20.0))
    print("[video] total frames:", total_frames, "| fps:", fps_in)

    if ASK_RANGE:
        START_FRAME = int(input("Start frame index (0..): ").strip())
        END_FRAME   = int(input("End frame index (inclusive): ").strip())
    else:
        START_FRAME = int(START_FRAME_FIXED)
        END_FRAME   = int(END_FRAME_FIXED)

    START_FRAME = max(0, START_FRAME)
    END_FRAME = min(total_frames-1, END_FRAME) if total_frames > 0 else END_FRAME
    if END_FRAME < START_FRAME:
        cap.release()
        raise RuntimeError("END_FRAME >= START_FRAME olmalı")

    n = int(input("Kaç nokta (marker) takip edilecek? (N): ").strip())
    if n <= 0:
        cap.release()
        raise RuntimeError("N>0 olmalı")

    cap.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
    ret, frame0 = cap.read()
    if not ret:
        cap.release()
        raise RuntimeError("Start frame okunamadı")

    top0, bot0 = split_top_bottom(frame0)
    frameL0 = maybe_upscale(top0, UPSCALE_TO_H)
    frameR0 = maybe_upscale(bot0, UPSCALE_TO_H)

    if frameL0.shape[:2] != frameR0.shape[:2]:
        cap.release()
        raise RuntimeError("Split sonrası L/R boyut uyuşmuyor")

    frameSize_cur = (frameL0.shape[1], frameL0.shape[0])
    print("[info] current view frameSize:", frameSize_cur)
    if frameSize_cur != frameSize_calib:
        print("[WARN] calib frameSize != current frameSize")

    # ---- ROI init ----
    print("\n[INIT ROI] ÜST frame (SOL) ROI seçimi\n")
    roisL0 = []
    for i in range(n):
        r = cv.selectROI(f"TOP(LEFT) init ROI #{i+1}", frameL0,
                         fromCenter=False, showCrosshair=True)
        cv.destroyWindow(f"TOP(LEFT) init ROI #{i+1}")
        if r is None or r[2] == 0 or r[3] == 0:
            cap.release()
            raise RuntimeError("ROI iptal/boş.")
        roisL0.append(tuple(map(int, r)))

    print("\n[INIT ROI] ALT frame (SAĞ) ROI seçimi\n")
    roisR0 = []
    for i in range(n):
        r = cv.selectROI(f"BOTTOM(RIGHT) init ROI #{i+1}", frameR0,
                         fromCenter=False, showCrosshair=True)
        cv.destroyWindow(f"BOTTOM(RIGHT) init ROI #{i+1}")
        if r is None or r[2] == 0 or r[3] == 0:
            cap.release()
            raise RuntimeError("ROI iptal/boş.")
        roisR0.append(tuple(map(int, r)))

    # ---- Init last positions ----
    lastL = np.full((n, 2), np.nan, dtype=np.float32)
    lastR = np.full((n, 2), np.nan, dtype=np.float32)
    lastAreaL = np.full((n,), np.nan, dtype=np.float32)
    lastAreaR = np.full((n,), np.nan, dtype=np.float32)

    for i in range(n):
        cL = red_centroids_in_roi(frameL0, roisL0[i], used_pts=None,
                                  used_min_dist=0, relax=1.0)
        cR = red_centroids_in_roi(frameR0, roisR0[i], used_pts=None,
                                  used_min_dist=0, relax=1.0)
        if len(cL) > 0:
            cx, cy, a = pick_nearest_with_area(cL, roi_center(roisL0[i]),
                                               last_area=None, area_alpha=0)
            lastL[i] = (cx, cy); lastAreaL[i] = a
        else:
            lastL[i] = roi_center(roisL0[i])

        if len(cR) > 0:
            cx, cy, a = pick_nearest_with_area(cR, roi_center(roisR0[i]),
                                               last_area=None, area_alpha=0)
            lastR[i] = (cx, cy); lastAreaR[i] = a
        else:
            lastR[i] = roi_center(roisR0[i])

    # ---- Init Kalman filters ----
    kalman_L = [make_kalman() for _ in range(n)]
    kalman_R = [make_kalman() for _ in range(n)]
    kalman_init_done_L = np.zeros(n, dtype=bool)
    kalman_init_done_R = np.zeros(n, dtype=bool)

    for i in range(n):
        if np.isfinite(lastL[i, 0]):
            kalman_init(kalman_L[i], lastL[i, 0], lastL[i, 1])
            kalman_init_done_L[i] = True
        if np.isfinite(lastR[i, 0]):
            kalman_init(kalman_R[i], lastR[i, 0], lastR[i, 1])
            kalman_init_done_R[i] = True

    missL = np.zeros((n,), dtype=np.int32)
    missR = np.zeros((n,), dtype=np.int32)

    # ---- Grayscale frames for optical flow ----
    prevGrayL = cv.cvtColor(frameL0, cv.COLOR_BGR2GRAY)
    prevGrayR = cv.cvtColor(frameR0, cv.COLOR_BGR2GRAY)

    # ---- Video writers ----
    Hv, Wv = frameL0.shape[0], frameL0.shape[1]
    out_size = (Wv * 2, Hv)
    fps_out = fps_in if USE_INPUT_FPS else float(OUT_FPS)

    writer_mp4 = cv.VideoWriter(OUT_MP4, cv.VideoWriter_fourcc(*"mp4v"),
                                 float(fps_out), out_size)
    if not writer_mp4.isOpened():
        cap.release()
        raise RuntimeError("MP4 VideoWriter açılamadı.")

    writer_avi = None
    if WRITE_AVI:
        writer_avi, avi_codec = open_video_writer_avi(OUT_AVI, fps_out,
                                                       out_size, AVI_CODECS)
        if writer_avi is None:
            print("[WARN] AVI writer açılamadı.")
        else:
            print(f"[avi] enabled -> {OUT_AVI} | codec={avi_codec}")

    # ---- Storage ----
    num_frames = END_FRAME - START_FRAME + 1

    UVL = np.full((num_frames, n, 2), np.nan, dtype=np.float32)
    UVR = np.full((num_frames, n, 2), np.nan, dtype=np.float32)
    OKL = np.zeros((num_frames, n), dtype=np.uint8)
    OKR = np.zeros((num_frames, n), dtype=np.uint8)
    OK  = np.zeros((num_frames, n), dtype=np.uint8)
    MODEL = np.empty((num_frames, n), dtype=object)
    MODER = np.empty((num_frames, n), dtype=object)

    XYZ_marker = np.full((num_frames, n, 3), np.nan, dtype=np.float64)

    TIPL    = np.full((num_frames, n, 2), np.nan, dtype=np.float32)
    TIPR    = np.full((num_frames, n, 2), np.nan, dtype=np.float32)
    ANGL_2D = np.full((num_frames, n), np.nan, dtype=np.float32)
    ANGR_2D = np.full((num_frames, n), np.nan, dtype=np.float32)
    OKTL    = np.zeros((num_frames, n), dtype=np.uint8)
    OKTR    = np.zeros((num_frames, n), dtype=np.uint8)

    XYZ_tip  = np.full((num_frames, n, 3), np.nan, dtype=np.float64)
    ANG_PHYS = np.full((num_frames, n), np.nan, dtype=np.float32)
    OK_PHYS  = np.zeros((num_frames, n), dtype=np.uint8)

    SEARCHBOXL = np.full((num_frames, n, 4), -1, dtype=np.int32)
    SEARCHBOXR = np.full((num_frames, n, 4), -1, dtype=np.int32)

    # ---- Ablation counters ----
    # Track how many times Kalman sustained a track beyond gap-hold
    kalman_sustained_L = 0
    kalman_sustained_R = 0
    lk_used_L = 0
    lk_used_R = 0

    cap.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)

    for t in range(num_frames):
        fidx = START_FRAME + t
        ret, frame = cap.read()
        if not ret:
            print("[WARN] Frame okunamadı:", fidx)
            break

        top, bot = split_top_bottom(frame)
        frameL = maybe_upscale(top, UPSCALE_TO_H)
        frameR = maybe_upscale(bot, UPSCALE_TO_H)
        rawL = frameL.copy()
        rawR = frameR.copy()
        drawL = frameL.copy()
        drawR = frameR.copy()

        currGrayL = cv.cvtColor(frameL, cv.COLOR_BGR2GRAY)
        currGrayR = cv.cvtColor(frameR, cv.COLOR_BGR2GRAY)

        # ---- Optical flow prediction for all markers ----
        valid_lastL = np.array([
            lastL[i] if np.isfinite(lastL[i, 0]) else np.array([np.nan, np.nan])
            for i in range(n)
        ], dtype=np.float32)
        valid_lastR = np.array([
            lastR[i] if np.isfinite(lastR[i, 0]) else np.array([np.nan, np.nan])
            for i in range(n)
        ], dtype=np.float32)

        lk_predL, lk_validL = optical_flow_predict(prevGrayL, currGrayL, valid_lastL)
        lk_predR, lk_validR = optical_flow_predict(prevGrayR, currGrayR, valid_lastR)

        ptsL = np.full((n, 2), np.nan, dtype=np.float32)
        ptsR = np.full((n, 2), np.nan, dtype=np.float32)
        okL_arr = np.zeros((n,), dtype=np.uint8)
        okR_arr = np.zeros((n,), dtype=np.uint8)
        okLR    = np.zeros((n,), dtype=np.uint8)
        roisL_new = [None] * n
        roisR_new = [None] * n
        modesL = ["lost"] * n
        modesR = ["lost"] * n
        usedL, usedR = [], []

        tipL_frame   = np.full((n, 2), np.nan, dtype=np.float32)
        tipR_frame   = np.full((n, 2), np.nan, dtype=np.float32)
        angL_frame   = np.full((n,), np.nan, dtype=np.float32)
        angR_frame   = np.full((n,), np.nan, dtype=np.float32)
        okTL_frame   = np.zeros((n,), dtype=np.uint8)
        okTR_frame   = np.zeros((n,), dtype=np.uint8)
        searchL_frame = np.full((n, 4), -1, dtype=np.int32)
        searchR_frame = np.full((n, 4), -1, dtype=np.int32)

        for i in range(n):
            # ---- Kalman predict ----
            if USE_KALMAN and kalman_init_done_L[i]:
                kx, ky = kalman_predict(kalman_L[i])
                kalman_pred_L = np.array([kx, ky], dtype=np.float32)
            else:
                kalman_pred_L = None

            if USE_KALMAN and kalman_init_done_R[i]:
                kx, ky = kalman_predict(kalman_R[i])
                kalman_pred_R = np.array([kx, ky], dtype=np.float32)
            else:
                kalman_pred_R = None

            # ---- Choose ROI center: LK > last_xy ----
            pred_for_roi_L = lk_predL[i] if (USE_OPTICAL_FLOW and lk_validL[i]) else None
            pred_for_roi_R = lk_predR[i] if (USE_OPTICAL_FLOW and lk_validR[i]) else None

            if USE_OPTICAL_FLOW and lk_validL[i]:
                lk_used_L += 1
            if USE_OPTICAL_FLOW and lk_validR[i]:
                lk_used_R += 1

            # ---- LEFT marker tracking ----
            newL, roiL_used, okL, modeL, areaL = track_roi_then_2x(
                rawL, lastL[i], lastAreaL[i], BASE_ROI_SIZE,
                predicted_xy=pred_for_roi_L,
                used_pts=usedL, used_min_dist=USED_MIN_DIST
            )
            roisL_new[i] = roiL_used
            modesL[i] = modeL
            okL_arr[i] = 1 if okL else 0

            if okL:
                # Kalman correct with detection
                if USE_KALMAN:
                    cx, cy = kalman_correct(kalman_L[i], float(newL[0]), float(newL[1]))
                    newL = np.array([cx, cy], dtype=np.float32)
                    kalman_init_done_L[i] = True
                lastL[i] = newL
                lastAreaL[i] = areaL
                ptsL[i] = newL
                missL[i] = 0
                usedL.append([float(newL[0]), float(newL[1])])
            else:
                missL[i] += 1
                if missL[i] <= GAP_HOLD and np.isfinite(lastL[i, 0]):
                    ptsL[i] = lastL[i]
                    modesL[i] = "hold"
                elif USE_KALMAN and kalman_init_done_L[i] and kalman_pred_L is not None:
                    # Kalman sustains track beyond gap-hold
                    ptsL[i] = kalman_pred_L
                    modesL[i] = "kalman"
                    lastL[i] = kalman_pred_L
                    kalman_sustained_L += 1
                else:
                    if SET_NAN_AFTER_GAP:
                        lastL[i] = np.array([np.nan, np.nan], dtype=np.float32)
                        lastAreaL[i] = np.nan

            # ---- RIGHT marker tracking ----
            newR, roiR_used, okR, modeR, areaR = track_roi_then_2x(
                rawR, lastR[i], lastAreaR[i], BASE_ROI_SIZE,
                predicted_xy=pred_for_roi_R,
                used_pts=usedR, used_min_dist=USED_MIN_DIST
            )
            roisR_new[i] = roiR_used
            modesR[i] = modeR
            okR_arr[i] = 1 if okR else 0

            if okR:
                if USE_KALMAN:
                    cx, cy = kalman_correct(kalman_R[i], float(newR[0]), float(newR[1]))
                    newR = np.array([cx, cy], dtype=np.float32)
                    kalman_init_done_R[i] = True
                lastR[i] = newR
                lastAreaR[i] = areaR
                ptsR[i] = newR
                missR[i] = 0
                usedR.append([float(newR[0]), float(newR[1])])
            else:
                missR[i] += 1
                if missR[i] <= GAP_HOLD and np.isfinite(lastR[i, 0]):
                    ptsR[i] = lastR[i]
                    modesR[i] = "hold"
                elif USE_KALMAN and kalman_init_done_R[i] and kalman_pred_R is not None:
                    ptsR[i] = kalman_pred_R
                    modesR[i] = "kalman"
                    lastR[i] = kalman_pred_R
                    kalman_sustained_R += 1
                else:
                    if SET_NAN_AFTER_GAP:
                        lastR[i] = np.array([np.nan, np.nan], dtype=np.float32)
                        lastAreaR[i] = np.nan

            okLR[i] = 1 if (okL and okR) else 0

            # ---- Tuft detection ----
            if np.isfinite(ptsL[i, 0]) and np.isfinite(ptsL[i, 1]):
                tip, ang2d, ok_tuft, search_box, _ = find_local_tuft_with_reacquire(
                    rawL, ptsL[i], ref_axis=ANGLE_REF)
                tipL_frame[i] = tip
                angL_frame[i] = ang2d
                okTL_frame[i] = 1 if ok_tuft else 0
                if search_box is not None:
                    searchL_frame[i] = np.array(search_box, dtype=np.int32)

            if np.isfinite(ptsR[i, 0]) and np.isfinite(ptsR[i, 1]):
                tip, ang2d, ok_tuft, search_box, _ = find_local_tuft_with_reacquire(
                    rawR, ptsR[i], ref_axis=ANGLE_REF)
                tipR_frame[i] = tip
                angR_frame[i] = ang2d
                okTR_frame[i] = 1 if ok_tuft else 0
                if search_box is not None:
                    searchR_frame[i] = np.array(search_box, dtype=np.int32)

        # ---- Triangulate markers ----
        xyz_marker_frame = np.full((n, 3), np.nan, dtype=np.float64)
        valid_m = okLR.astype(bool)
        if np.any(valid_m):
            ptsL_rect = rectified_points_from_pixel(ptsL[valid_m], K1, dist1, R1, P1)
            ptsR_rect = rectified_points_from_pixel(ptsR[valid_m], K2, dist2, R2, P2)
            xyz_valid = triangulate_rectified(P1, P2, ptsL_rect, ptsR_rect)
            xyz_marker_frame[valid_m] = xyz_valid

        # ---- Triangulate tips + physical angle ----
        xyz_tip_frame  = np.full((n, 3), np.nan, dtype=np.float64)
        ang_phys_frame = np.full((n,), np.nan, dtype=np.float32)
        ok_phys_frame  = np.zeros((n,), dtype=np.uint8)

        for i in range(n):
            if not (valid_m[i] and okTL_frame[i] == 1 and okTR_frame[i] == 1):
                continue
            tipL_px = tipL_frame[i]
            tipR_px = tipR_frame[i]
            if not (np.all(np.isfinite(tipL_px)) and np.all(np.isfinite(tipR_px))):
                continue
            tipL_rect = rectified_points_from_pixel(tipL_px.reshape(1, 2), K1, dist1, R1, P1)
            tipR_rect = rectified_points_from_pixel(tipR_px.reshape(1, 2), K2, dist2, R2, P2)
            tip_xyz = triangulate_rectified(P1, P2, tipL_rect, tipR_rect)[0]
            xyz_tip_frame[i] = tip_xyz
            aphys = physical_angle_xy(xyz_marker_frame[i], tip_xyz, ref_axis=ANGLE_REF)
            if np.isfinite(aphys):
                ang_phys_frame[i] = float(aphys)
                ok_phys_frame[i] = 1

        # ---- Draw overlays ----
        for i in range(n):
            xyz_to_show = xyz_marker_frame[i] if valid_m[i] else None
            draw_overlay_one(drawL, roisL_new[i], ptsL[i], i, bool(okL_arr[i]),
                             modesL[i], xyz=xyz_to_show)
            draw_overlay_one(drawR, roisR_new[i], ptsR[i], i, bool(okR_arr[i]),
                             modesR[i], xyz=xyz_to_show)

            txt_phys = None
            if SHOW_PHYS_ANGLE and ok_phys_frame[i] == 1:
                axis = "X" if str(ANGLE_REF).upper() == "X" else "Y"
                txt_phys = f"angXY@{axis} {ang_phys_frame[i]:+.1f} deg"

            if okTL_frame[i] == 1:
                draw_tuft_overlay(drawL, ptsL[i], tipL_frame[i], True,
                    search_box=tuple(searchL_frame[i]) if searchL_frame[i, 0] >= 0 else None,
                    text=txt_phys)
            if okTR_frame[i] == 1:
                draw_tuft_overlay(drawR, ptsR[i], tipR_frame[i], True,
                    search_box=tuple(searchR_frame[i]) if searchR_frame[i, 0] >= 0 else None,
                    text=txt_phys)

            if SHOW_PHYS_ANGLE and ok_phys_frame[i] == 1:
                axis = "X" if str(ANGLE_REF).upper() == "X" else "Y"
                txt = f"angXY@{axis} {ang_phys_frame[i]:+.1f} deg"
                if okL_arr[i] == 1 and np.isfinite(ptsL[i, 0]):
                    uL, vL = int(round(ptsL[i, 0])), int(round(ptsL[i, 1]))
                    cv.putText(drawL, txt, (uL+6, vL+40),
                               cv.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2)
                if okR_arr[i] == 1 and np.isfinite(ptsR[i, 0]):
                    uR, vR = int(round(ptsR[i, 0])), int(round(ptsR[i, 1]))
                    cv.putText(drawR, txt, (uR+6, vR+40),
                               cv.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2)

        cv.putText(drawL, f"Frame {fidx}", (10, 25),
                   cv.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv.putText(drawR, f"Frame {fidx}", (10, 25),
                   cv.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        stacked_out = np.ascontiguousarray(np.hstack([drawL, drawR]))
        writer_mp4.write(stacked_out)
        if writer_avi is not None:
            writer_avi.write(stacked_out)

        # ---- Store ----
        UVL[t] = ptsL; UVR[t] = ptsR
        OKL[t] = okL_arr; OKR[t] = okR_arr; OK[t] = okLR
        for i in range(n):
            MODEL[t, i] = modesL[i]
            MODER[t, i] = modesR[i]
        XYZ_marker[t] = xyz_marker_frame
        TIPL[t] = tipL_frame; TIPR[t] = tipR_frame
        ANGL_2D[t] = angL_frame; ANGR_2D[t] = angR_frame
        OKTL[t] = okTL_frame; OKTR[t] = okTR_frame
        XYZ_tip[t] = xyz_tip_frame
        ANG_PHYS[t] = ang_phys_frame; OK_PHYS[t] = ok_phys_frame
        SEARCHBOXL[t] = searchL_frame; SEARCHBOXR[t] = searchR_frame

        # Update previous grayscale
        prevGrayL = currGrayL
        prevGrayR = currGrayR

        if (t % 20) == 0:
            print(f"[proc] frame {fidx} ({t+1}/{num_frames}) | "
                  f"LK_used_L={lk_used_L} KalmanSust_L={kalman_sustained_L}")

    cap.release()
    writer_mp4.release()
    if writer_avi is not None:
        writer_avi.release()

    # ---- Print ablation summary ----
    total_marker_frames = num_frames * n
    print(f"\n[ABLATION SUMMARY]")
    print(f"  Total marker-frames processed : {total_marker_frames}")
    print(f"  LK flow used (L / R)          : {lk_used_L} / {lk_used_R}")
    print(f"  Kalman sustained beyond hold  : {kalman_sustained_L} / {kalman_sustained_R}")
    print(f"  LK usage rate (L)             : {100*lk_used_L/max(total_marker_frames,1):.1f}%")
    print(f"  Kalman sustain rate (L)       : {100*kalman_sustained_L/max(total_marker_frames,1):.1f}%")

    # ---- Export CSV ----
    with open(OUT_CSV, "w", encoding="utf-8") as f:
        f.write(
            "frame,marker,okLR,okL,okR,modeL,modeR,"
            "uL,vL,uR,vR,Xm,Ym,Zm,"
            "okTuftL,tipLu,tipLv,angL2D_deg,searchL_x,searchL_y,searchL_w,searchL_h,"
            "okTuftR,tipRu,tipRv,angR2D_deg,searchR_x,searchR_y,searchR_w,searchR_h,"
            "Xt,Yt,Zt,okPhys,angPhysXY_deg,angRef\n"
        )
        axis = "X" if str(ANGLE_REF).upper() == "X" else "Y"
        for t in range(num_frames):
            fidx = START_FRAME + t
            for i in range(n):
                f.write(
                    f"{fidx},{i+1},{int(OK[t,i])},{int(OKL[t,i])},{int(OKR[t,i])},"
                    f"{MODEL[t,i]},{MODER[t,i]},"
                    f"{UVL[t,i,0]:.3f},{UVL[t,i,1]:.3f},"
                    f"{UVR[t,i,0]:.3f},{UVR[t,i,1]:.3f},"
                    f"{XYZ_marker[t,i,0]:.6f},{XYZ_marker[t,i,1]:.6f},{XYZ_marker[t,i,2]:.6f},"
                    f"{int(OKTL[t,i])},{TIPL[t,i,0]:.3f},{TIPL[t,i,1]:.3f},"
                    f"{float(ANGL_2D[t,i]):.3f},"
                    f"{SEARCHBOXL[t,i,0]},{SEARCHBOXL[t,i,1]},"
                    f"{SEARCHBOXL[t,i,2]},{SEARCHBOXL[t,i,3]},"
                    f"{int(OKTR[t,i])},{TIPR[t,i,0]:.3f},{TIPR[t,i,1]:.3f},"
                    f"{float(ANGR_2D[t,i]):.3f},"
                    f"{SEARCHBOXR[t,i,0]},{SEARCHBOXR[t,i,1]},"
                    f"{SEARCHBOXR[t,i,2]},{SEARCHBOXR[t,i,3]},"
                    f"{XYZ_tip[t,i,0]:.6f},{XYZ_tip[t,i,1]:.6f},{XYZ_tip[t,i,2]:.6f},"
                    f"{int(OK_PHYS[t,i])},{float(ANG_PHYS[t,i]):.3f},{axis}\n"
                )

    # ---- Export NPZ ----
    np.savez(
        OUT_NPZ,
        stacked_video=STACKED_VIDEO, calib_npz=CALIB_NPZ,
        start_frame=int(START_FRAME), end_frame=int(END_FRAME),
        N=int(n), ANGLE_REF=str(ANGLE_REF),
        UVL_px=UVL, UVR_px=UVR,
        OKL=OKL, OKR=OKR, OK=OK,
        modeL=MODEL, modeR=MODER,
        XYZ_marker_mm=XYZ_marker,
        TIPL_px=TIPL, TIPR_px=TIPR,
        ANGL2D_deg=ANGL_2D, ANGR2D_deg=ANGR_2D,
        OKTL=OKTL, OKTR=OKTR,
        XYZ_tip_mm=XYZ_tip,
        ANG_physXY_deg=ANG_PHYS, OK_phys=OK_PHYS,
        search_box_L=SEARCHBOXL, search_box_R=SEARCHBOXR,
        frameSize_current=np.array(frameSize_cur, dtype=np.int32),
        frameSize_calib=np.array(frameSize_calib, dtype=np.int32),
        upscale_to_h=UPSCALE_TO_H,
        fps_in=float(fps_in), fps_out=float(fps_out),
        # Motion module params
        use_optical_flow=USE_OPTICAL_FLOW,
        use_kalman=USE_KALMAN,
        lk_win_size=np.array(LK_WIN_SIZE),
        lk_max_level=int(LK_MAX_LEVEL),
        kalman_proc_noise=float(KALMAN_PROC_NOISE),
        kalman_meas_noise=float(KALMAN_MEAS_NOISE),
        # Ablation stats
        lk_used_L=int(lk_used_L), lk_used_R=int(lk_used_R),
        kalman_sustained_L=int(kalman_sustained_L),
        kalman_sustained_R=int(kalman_sustained_R),
        total_marker_frames=int(total_marker_frames),
    )

    print(f"\n[save] {OUT_CSV}")
    print(f"[save] {OUT_NPZ}")
    print(f"[save] {OUT_MP4}")
    if WRITE_AVI and os.path.exists(OUT_AVI):
        print(f"[save] {OUT_AVI}")
    print("\nDONE.")

if __name__ == "__main__":
    main()

[calib] loaded | frameSize(each view w,h): (1280, 360)
[video] total frames: 686 | fps: 59.94005994005994
Start frame index (0..): 5
End frame index (inclusive): 200
Kaç nokta (marker) takip edilecek? (N): 5
[info] current view frameSize: (3840, 1080)
[WARN] calib frameSize != current frameSize

[INIT ROI] ÜST frame (SOL) ROI seçimi


[INIT ROI] ALT frame (SAĞ) ROI seçimi

[avi] enabled -> tracked_xyz_stacked_localtuft_original_tuft\tracked_overlay_CFD_Merged_Seyir2_localtuft_kalman_lk.avi | codec=MJPG


C:\Users\edanu\AppData\Local\Temp\ipykernel_26836\4257633656.py:210: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(pred[0]), float(pred[1])
C:\Users\edanu\AppData\Local\Temp\ipykernel_26836\4257633656.py:216: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(corr[0]), float(corr[1])


[proc] frame 5 (1/196) | LK_used_L=5 KalmanSust_L=0
[proc] frame 25 (21/196) | LK_used_L=105 KalmanSust_L=0
[proc] frame 45 (41/196) | LK_used_L=205 KalmanSust_L=0
[proc] frame 65 (61/196) | LK_used_L=305 KalmanSust_L=0
[proc] frame 85 (81/196) | LK_used_L=405 KalmanSust_L=0
[proc] frame 105 (101/196) | LK_used_L=505 KalmanSust_L=0
[proc] frame 125 (121/196) | LK_used_L=605 KalmanSust_L=0
[proc] frame 145 (141/196) | LK_used_L=705 KalmanSust_L=0
[proc] frame 165 (161/196) | LK_used_L=805 KalmanSust_L=0
[proc] frame 185 (181/196) | LK_used_L=905 KalmanSust_L=0

[ABLATION SUMMARY]
  Total marker-frames processed : 980
  LK flow used (L / R)          : 980 / 980
  Kalman sustained beyond hold  : 0 / 0
  LK usage rate (L)             : 100.0%
  Kalman sustain rate (L)       : 0.0%

[save] tracked_xyz_stacked_localtuft_original_tuft\tracked_xyz_mm_CFD_Merged_Seyir2_localtuft_kalman_lk.csv
[save] tracked_xyz_stacked_localtuft_original_tuft\tracked_xyz_mm_CFD_Merged_Seyir2_localtuft_kalman_lk